In [1]:
import os
import sys
import requests
import datetime as dt
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.rci import Rci

# CorporateFinanceData 動作確認
`CorporateFinanceData` を使って財務データ取得メソッドを個別セルで実行します。

In [2]:
# APIの接続先（notebookコンテナでは通常 http://api:8000 が設定される）
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
# データを取得する関数
sys.path.append("/workspace/notebook")

### エンドポイントへRequestし財務データを更新する

In [3]:
# Vestas Wind Systems (Yahoo Finance: VWS.CO)
code = "VWS"
market = "CO"
"""
post_payload = {
    "code": code,
    "market": market,
}

# /api/v1/corp_finance_data/
post_url = f"{API_BASE_URL}/api/v1/corp_finance_data/"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())
"""

'\npost_payload = {\n    "code": code,\n    "market": market,\n}\n\n# /api/v1/corp_finance_data/\npost_url = f"{API_BASE_URL}/api/v1/corp_finance_data/"\npost_response = requests.post(post_url, json=post_payload, timeout=60)\n\nprint("POST URL:", post_url)\nprint("POST status:", post_response.status_code)\nprint("POST response:", post_response.json())\n'

### 直近４年分の財務諸表を取得

In [4]:
get_params = {
    "code": code,
    "market": market,
}
# /api/v1/corp_finance_data/financials/
get_url = f"{API_BASE_URL}/api/v1/corp_finance_data/financials/"
get_response = requests.get(get_url, params=get_params, timeout=60)

if get_response.status_code != 200:
    print("GET URL:", get_url)
    print("GET status:", get_response.status_code)
    print("GET response:", get_response.json())
else:
    financials_df = pd.DataFrame(get_response.json().get("results", []))
    financials_df.head()


### 直近４年分のバランスシートを取得

In [5]:
get_params = {
    "code": code,
    "market": market,
}
# /api/v1/corp_finance_data/balance_sheet/
get_url = f"{API_BASE_URL}/api/v1/corp_finance_data/balance_sheet/"
get_response = requests.get(get_url, params=get_params, timeout=60)

if get_response.status_code != 200:
    print("GET URL:", get_url)
    print("GET status:", get_response.status_code)
    print("GET response:", get_response.json())

GET URL: http://api:8000/api/v1/corp_finance_data/balance_sheet/
GET status: 500
GET response: {'detail': 'An unexpected error occurred'}


In [6]:
balance_sheet_df = pd.DataFrame(get_response.json().get("results", []))
balance_sheet_df.head()

""


### 直近４年分のキャッシュフローを取得

In [7]:
get_params = {
    "code": code,
    "market": market,
}
# /api/v1/corp_finance_data/cash_flow/
get_url = f"{API_BASE_URL}/api/v1/corp_finance_data/cash_flow/"
get_response = requests.get(get_url, params=get_params, timeout=60)

if get_response.status_code != 200:
    print("GET URL:", get_url)
    print("GET status:", get_response.status_code)
    print("GET response:", get_response.json())

In [8]:
cash_flow_df = pd.DataFrame(get_response.json().get("results", []))
cash_flow_df.head()

,date,market,code,free_cash_flow,repurchase_of_capital_stock,repayment_of_debt,issuance_of_debt,capital_expenditure,end_cash_position,beginning_cash_position,...,provisionand_write_offof_assets,deferred_tax,depreciation_and_amortization,amortization_cash_flow,depreciation,net_foreign_currency_exchange_gain_loss,gain_loss_on_sale_of_ppe,net_income_from_continuing_operations,created_at,updated_at
0,2021-12-31,CO,VWS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,9.820000e+08,NaN,6000000.0,NaN,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
1,2022-12-31,CO,VWS,-1.014000e+09,0.0,-713000000.0,1.756000e+09,-8.190000e+08,2.378000e+09,2.420000e+09,...,420000000.0,-146000000.0,1.088000e+09,NaN,1.088000e+09,NaN,-8000000.0,-1.572000e+09,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
2,2023-12-31,CO,VWS,1.350000e+08,-11000000.0,-221000000.0,1.137000e+09,-8.920000e+08,3.318000e+09,2.378000e+09,...,236000000.0,24000000.0,7.890000e+08,7.890000e+08,7.890000e+08,171000000.0,0.0,7.800000e+07,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
3,2024-12-31,CO,VWS,1.177000e+09,-40000000.0,-345000000.0,8.400000e+07,-1.155000e+09,3.817000e+09,3.318000e+09,...,215000000.0,211000000.0,8.620000e+08,8.620000e+08,NaN,172000000.0,-13000000.0,4.940000e+08,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
4,2025-12-31,CO,VWS,1.008000e+09,-282000000.0,-122000000.0,1.500000e+08,-1.278000e+09,4.384000e+09,3.817000e+09,...,-157000000.0,259000000.0,1.059000e+09,1.059000e+09,NaN,NaN,NaN,7.800000e+08,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z


### 直近４年分の収益を取得

In [9]:
get_params = {
    "code": code,
    "market": market,
}
# /api/v1/corp_finance_data/earnings/
get_url = f"{API_BASE_URL}/api/v1/corp_finance_data/earnings/"
get_response = requests.get(get_url, params=get_params, timeout=60)

if get_response.status_code != 200:
    print("GET URL:", get_url)
    print("GET status:", get_response.status_code)
    print("GET response:", get_response.json())

In [10]:
earnings_df = pd.DataFrame(get_response.json().get("results", []))
earnings_df.head()

,date,market,code,revenue,earnings,created_at,updated_at
0,2021-12-31,CO,VWS,NaN,NaN,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
1,2022-12-31,CO,VWS,1.448600e+10,-1.572000e+09,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
2,2023-12-31,CO,VWS,1.538200e+10,7.700000e+07,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
3,2024-12-31,CO,VWS,1.729500e+10,4.990000e+08,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
4,2025-12-31,CO,VWS,1.882200e+10,7.780000e+08,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z


### 直近４年分の四半期収益を取得

In [11]:
get_params = {
    "code": code,
    "market": market,
}
# /api/v1/corp_finance_data/quarterly_earnings/
get_url = f"{API_BASE_URL}/api/v1/corp_finance_data/quarterly_earnings/"
get_response = requests.get(get_url, params=get_params, timeout=60)

if get_response.status_code != 200:
    print("GET URL:", get_url)
    print("GET status:", get_response.status_code)
    print("GET response:", get_response.json())

In [12]:
quarterly_earnings_df = pd.DataFrame(get_response.json().get("results", []))
quarterly_earnings_df.head()

,date,market,code,revenue,earnings,created_at,updated_at
0,2024-09-30,CO,VWS,NaN,NaN,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
1,2024-12-31,CO,VWS,6.141000e+09,598000000.0,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
2,2025-03-31,CO,VWS,3.468000e+09,5000000.0,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
3,2025-06-30,CO,VWS,3.745000e+09,32000000.0,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
4,2025-09-30,CO,VWS,5.339000e+09,302000000.0,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z


In [ ]:
# 直近2年の期間を作成
end_date = dt.date(2024, 12, 31)
start_date = end_date - dt.timedelta(days=365 * 2)

name = ""
code = "6728"
market = "TSE"

get_url = f"{API_BASE_URL}/api/v1/time_series_data"
get_params = {
    "code": code,
    "market": market,
    "start": start_date.isoformat(),
    "end": end_date.isoformat(),
}

get_response = requests.get(get_url, params=get_params, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_results = pd.DataFrame(results)

    print("取得件数:", len(df_results))
    display(df_results.head())
else:
    print("GET response:", get_response.json())

GET URL: http://api:8000/api/v1/time_series_data
GET status: 200
取得件数: 502


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,428958,TAN,NYSE,2023-01-03,71.748383,74.154243,71.320894,73.468279,489000,73.092491,...,81.265492,74.976390,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,428959,TAN,NYSE,2023-01-04,72.871788,73.001026,71.808039,72.086402,680400,72.480090,...,81.085060,74.465684,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,428960,TAN,NYSE,2023-01-05,70.495750,71.947220,70.356561,71.947220,676600,72.283246,...,80.907766,73.986039,False,NaN,NaN,NaN,NaN,-81.666667,NaN,False
3,428961,TAN,NYSE,2023-01-06,73.438461,73.667113,71.002769,72.175876,1200200,72.297164,...,80.663778,73.563544,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,428962,TAN,NYSE,2023-01-09,74.273552,75.605728,74.213905,74.551915,978300,72.845939,...,80.144516,73.405188,False,NaN,NaN,NaN,NaN,NaN,NaN,True


In [14]:
df = df_results.copy()

display_start_date = dt.date(2024, 1, 1)

# date列をインデックスに設定してスライス
df = df.set_index(pd.to_datetime(df["date"])).sort_index()
df = df[display_start_date:end_date]

# レイアウト設定
layout = {
        "height": 900,
        "title": {"x": 0.5, "text": f"{name or code} {code}"},
        "xaxis": {"rangeslider": {"visible": False}, "type": "category"},
        "yaxis1": {"domain": [.36, 1.0], "title": "価格（円）", "side": "left", "tickformat": ","},
        "yaxis2": {"domain": [.30, .36]},
        "yaxis3": {"domain": [.20, .295], "title": "MACD", "side": "right"},
        "yaxis4": {"domain": [.10, .195], "title": "RCI", "side": "right"},
        "yaxis5": {"domain": [.0, .095], "title": "Volume", "side": "right"},
        "plot_bgcolor": "light blue",
}

# データの設定
data = [
        go.Candlestick(
            yaxis="y1",
            x=df.index,
            open=df["open"],
            high=df["high"],
            low=df["low"],
            close=df["close"],
            increasing_line_color="red",
            decreasing_line_color="gray",
            name=code,
        ),
        go.Scatter(yaxis="y1", x=df.index, y=df["ma5"], name="MA5", line={"color": "royalblue", "width": 1.2}),
        go.Scatter(yaxis="y1", x=df.index, y=df["ma25"], name="MA25", line={"color": "lightseagreen", "width": 1.2}),
        go.Scatter(yaxis="y1", x=df.index, y=df["upper2"], name="", line={"color": "lavender", "width": 0}),
        go.Scatter(
            yaxis="y1",
            x=df.index,
            y=df["lower2"],
            name="BB",
            line={"color": "lavender", "width": 0},
            fill="tonexty",
            fillcolor="rgba(170,170,170,.2)",
        ),
        go.Scatter(yaxis="y3", x=df.index, y=df["macd"], name="macd", line={"color": "magenta", "width": 1}),
        go.Scatter(yaxis="y3", x=df.index, y=df["macd_signal"], name="signal", line={"color": "green", "width": 1}),
        go.Bar(yaxis="y3", x=df.index, y=df["hist"], name="histgram", marker={"color": "slategray"}),
        go.Scatter(yaxis="y4", x=df.index, y=df["rci9"], name="RCI9", line={"color": "magenta", "width": 1}),
        go.Scatter(yaxis="y4", x=df.index, y=df["rci26"], name="RCI26", line={"color": "green", "width": 1}),
        go.Bar(yaxis="y5", x=df.index, y=df["volume"], name="Volume", marker=dict(color="slategray")),
]

# Figure作成
fig = go.Figure(data=data, layout=go.Layout(layout))

# 日付表示
fig.update_layout(
        {
            "xaxis": {
                "tickvals": df.index[::4],
                "ticktext": [x.strftime("%m-%d") for x in df.index[::4]],
            }
        }
)
fig.show()

/tmp/ipykernel_30/2401714417.py:7: Pandas4Warning: Slicing with a datetime.date object is deprecated. Explicitly cast to Timestamp instead.
  df = df[display_start_date:end_date]
